# Registration builds, side by side: centroid vs fit

The same patient, the same pair of studies and the same slice, read from two builds of the NYUMets
h5 set. For each pair:

| column | image |
|---|---|
| reference | the patient's reference study (never moved by either build) |
| other, OLD build | the other study as the old build stored it at that `slice_index` |
| other, NEW build | the same study as the new build stored it |
| the two difference maps | \|reference - other\| for each build, on one scale |

Rows are T1 and CT1. Images are `img_raw` divided by the NYUMets scales, on a fixed 0..1 window.
The number over each "other" panel is the masked T1 (or CT1) correlation with the reference on
that slice.

Read-only: nothing is written to either dataset.

## Knobs

In [ ]:
import os, glob, csv
import numpy as np
import torch
import h5py
import matplotlib.pyplot as plt
%matplotlib inline

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if os.path.basename(os.getcwd()) != "notebooks":
    REPO_ROOT = os.getcwd()
import sys
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from visualization.image import set_display_orient, subplot_images
set_display_orient("radiological")          # display only; stored pixels are canonical RAS

OLD_ROOT = "~/scratch/datasets/NYUMets_h5"            # the previous build (centroid)
NEW_ROOT = "~/scratch/datasets/NYUMets_h5_fit_test"   # the new build (fit)
OLD_NAME, NEW_NAME = "centroid", "fit"

SCALES = [2800.0, 2500.0, 2390.0, 2370.0]   # FLAIR, T1, CT1, T2 -- the NYUMets config scales
VMIN, VMAX = 0.0, 1.0
SLICE_RANGE = (50, 90)                      # original slice indices to draw from (what training uses)
N_PAIRS = 4                                 # study pairs per draw
N_SLICES = 2                                # slices shown per pair
SEED = None                                 # None = a fresh draw every run (printed); set to replay

# Optional: pick the pairs the diagnostic scored WORST on the old build instead of at random.
# Point this at the --csv of scripts/diagnose_nyumets_registration.py run on OLD_ROOT.
WORST_FROM_CSV = None                       # e.g. "centroid_diag.csv"

# Optional: name pairs yourself, as (patient, other_session). Overrides both of the above.
PAIRS = None                                # e.g. [("10178949", "6214216661")]

T1, CT1 = 1, 2
OLD_ROOT, NEW_ROOT = (os.path.abspath(os.path.expanduser(p)) for p in (OLD_ROOT, NEW_ROOT))
for p in (OLD_ROOT, NEW_ROOT):
    assert os.path.isdir(p), "not a directory: " + p
print("old:", OLD_ROOT)
print("new:", NEW_ROOT)

## Index both builds

Only patients present in BOTH builds with at least two studies in each can be compared. The
reference study is the one each build recorded (`reg_reference`); a pair is used only when both
builds agree on it, so the "reference" column is the same image in both.

In [ ]:
def index(root):
    """{patient: {session: path}} for every <case>/<case>_img.h5 under root. Attrs only."""
    out = {}
    for p in sorted(glob.glob(os.path.join(root, "*", "*_img.h5"))):
        with h5py.File(p, "r") as f:
            pid = f.attrs.get("patient", os.path.basename(p).split("_")[0])
            ses = f.attrs.get("session", "")
            pid = pid.decode() if isinstance(pid, bytes) else str(pid)
            ses = ses.decode() if isinstance(ses, bytes) else str(ses)
        out.setdefault(pid, {})[ses or os.path.basename(os.path.dirname(p))] = p
    return out


def attr(path, name, default=None):
    with h5py.File(path, "r") as f:
        v = f.attrs.get(name, default)
    return v.decode() if isinstance(v, bytes) else v


def reference_of(studies):
    """The session a build registered this patient onto, read from any of its files."""
    pid_ref = str(attr(next(iter(studies.values())), "reg_reference", ""))
    for ses in studies:
        if pid_ref.endswith("_" + ses) or pid_ref == ses:
            return ses
    return None


old, new = index(OLD_ROOT), index(NEW_ROOT)
CANDS = []                                  # (patient, reference session, other session)
skipped = 0
for pid in sorted(set(old) & set(new)):
    both = sorted(set(old[pid]) & set(new[pid]))
    if len(both) < 2:
        continue
    r_old, r_new = reference_of(old[pid]), reference_of(new[pid])
    if r_new is None or r_old != r_new or r_new not in both:
        skipped += 1
        continue
    CANDS += [(pid, r_new, s) for s in both if s != r_new]

print("%d patients in the old build, %d in the new, %d in both"
      % (len(old), len(new), len(set(old) & set(new))))
print("%d comparable (patient, other study) pairs; %d patient(s) skipped because the two builds "
      "used a different reference study" % (len(CANDS), skipped))
print("methods:  old =", attr(old[CANDS[0][0]][CANDS[0][2]], "reg_method", "?"),
      "  new =", attr(new[CANDS[0][0]][CANDS[0][2]], "reg_method", "?"))
assert CANDS, "no patient has the same reference and a second study in both builds"

## Draw and show

**Re-run this cell for a new draw.** Each figure is one study pair; each pair of rows (T1, CT1) is
one slice. The slice is addressed by its ORIGINAL index, so it names the same level of the
reference in both builds.

In [ ]:
def load_slice(path, z):
    """(T1, CT1, mask) at ORIGINAL slice index z, scaled; None if this file does not hold z."""
    with h5py.File(path, "r") as f:
        zi = np.asarray(f["slice_index"])
        hit = np.flatnonzero(zi == z)
        if hit.size == 0:
            return None
        k = int(hit[0])
        img = np.asarray(f["img_raw"][k], dtype=np.float32)
        m = np.asarray(f["mask"][k, :, :, 0]).astype(bool)
    return img[..., T1] / SCALES[T1], img[..., CT1] / SCALES[CT1], m


def shared_slices(paths):
    """Original slice indices every one of `paths` holds, inside SLICE_RANGE."""
    keep = None
    for p in paths:
        with h5py.File(p, "r") as f:
            zi = set(np.asarray(f["slice_index"]).tolist())
        keep = zi if keep is None else keep & zi
    lo, hi = SLICE_RANGE if SLICE_RANGE else (-10**9, 10**9)
    return sorted(z for z in keep if lo <= z < hi)


def ncc(a, b, m):
    if int(m.sum()) < 200:
        return float("nan")
    x, y = a[m] - a[m].mean(), b[m] - b[m].mean()
    return float((x * y).sum() / max(float(np.sqrt((x * x).sum() * (y * y).sum())), 1e-8))


def pick_pairs(rng):
    if PAIRS:
        want = {(str(p), str(s)) for p, s in PAIRS}
        got = [c for c in CANDS if (c[0], c[2]) in want]
        missing = want - {(c[0], c[2]) for c in got}
        if missing:
            print("not comparable in both builds, skipped:", sorted(missing))
        return got
    if WORST_FROM_CSV:
        with open(WORST_FROM_CSV, newline="") as f:
            rows = [r for r in csv.DictReader(f) if r.get("before") not in (None, "", "nan")]
        worst = {}
        for r in rows:                       # a study's score = its worst slice on the old build
            # the diagnostic compares against the patient's FIRST study, this notebook against
            # the build's registration reference, so credit the row to BOTH of its studies:
            # whichever is not the reference here is the "other" of a comparable pair
            for ses in (str(r["other"]), str(r["ref"])):
                key = (str(r["patient"]), ses)
                worst[key] = min(worst.get(key, 9.0), float(r["before"]))
        ranked = sorted((c for c in CANDS if (c[0], c[2]) in worst),
                        key=lambda c: worst[(c[0], c[2])])
        if not ranked:
            print("none of the CSV's pairs exist in both builds; drawing at random instead")
        else:
            return ranked[:N_PAIRS]
    return [CANDS[i] for i in rng.permutation(len(CANDS))[:N_PAIRS]]


def show_pair(pid, ref, oth, rng):
    paths = [new[pid][ref], old[pid][oth], new[pid][oth]]
    zs = shared_slices(paths)
    if not zs:
        print("%s: %s / %s share no slice in %s across the two builds" % (pid, ref, oth, SLICE_RANGE))
        return None
    zs = sorted(rng.choice(zs, size=min(N_SLICES, len(zs)), replace=False).tolist())
    rows, labels, titles, scores = [], [], None, []
    for z in zs:
        R, O, N = (load_slice(p, z) for p in paths)
        for c, name in ((0, "T1"), (1, "CT1")):
            mo, mn = R[2] & O[2], R[2] & N[2]
            so, sn = ncc(R[c], O[c], mo), ncc(R[c], N[c], mn)
            scores.append((z, name, so, sn))
            rows.append([torch.from_numpy(x) for x in
                         (R[c], O[c], N[c], np.abs(R[c] - O[c]) * R[2], np.abs(R[c] - N[c]) * R[2])])
            labels.append("%s  z=%d\n%s %.2f | %s %.2f" % (name, z, OLD_NAME, so, NEW_NAME, sn))
    subplot_images(
        rows, row_labels=labels,
        col_titles=["reference\n" + ref, "other, %s\n%s" % (OLD_NAME, oth),
                    "other, %s\n%s" % (NEW_NAME, oth),
                    "|ref - other|, " + OLD_NAME, "|ref - other|, " + NEW_NAME],
        cmap=["gray", "gray", "gray", "magma", "magma"],
        vmin=[VMIN, VMIN, VMIN, 0.0, 0.0], vmax=[VMAX, VMAX, VMAX, 0.5 * VMAX, 0.5 * VMAX],
        magnitude=False, panel_size=(2.7, 3.0),
        suptitle="%s   reference %s  <-  %s   (img_raw / scale)" % (pid, ref, oth), show=False)
    plt.show()
    return scores


seed = int(np.random.SeedSequence().entropy % 2**32) if SEED is None else int(SEED)
rng = np.random.default_rng(seed)
print("seed %d   (set SEED to replay this draw)" % seed)
DRAWN = []
for pid, ref, oth in pick_pairs(rng):
    sc = show_pair(pid, ref, oth, rng)
    if sc:
        DRAWN += [(pid, oth) + s for s in sc]

print("\n%-10s %-12s %4s %-4s %9s %9s %8s" % ("patient", "other", "z", "", OLD_NAME, NEW_NAME, "change"))
for pid, oth, z, name, so, sn in DRAWN:
    print("%-10s %-12s %4d %-4s %9.3f %9.3f %+8.3f" % (pid, oth, z, name, so, sn, sn - so))

## Across many pairs (numbers only)

The figures above are a handful of slices. This scores the masked T1 and CT1 correlation with the
reference on `N_SCORE` random (pair, slice) samples in both builds, on the SAME slices, so the two
columns are paired -- unlike two separate runs of the diagnostic script, which each draw their own
patients.

In [ ]:
N_SCORE = 200                    # (pair, slice) samples
score_rng = np.random.default_rng(0)

rows = []
order = score_rng.permutation(len(CANDS))
i = 0
while len(rows) < N_SCORE and i < 20 * N_SCORE:
    pid, ref, oth = CANDS[order[i % len(order)]]
    i += 1
    paths = [new[pid][ref], old[pid][oth], new[pid][oth]]
    zs = shared_slices(paths)
    if not zs:
        continue
    z = int(score_rng.choice(zs))
    R, O, N = (load_slice(p, z) for p in paths)
    rows.append((ncc(R[0], O[0], R[2] & O[2]), ncc(R[0], N[0], R[2] & N[2]),
                 ncc(R[1], O[1], R[2] & O[2]), ncc(R[1], N[1], R[2] & N[2])))
A = np.array(rows, dtype=float)
A = A[np.isfinite(A).all(1)]
print("%d paired (pair, slice) samples over %d comparable pairs" % (len(A), len(CANDS)))
print("%-6s %10s %10s %10s %22s" % ("", OLD_NAME, NEW_NAME, "median +", "new better on"))
for name, a, b in (("T1", A[:, 0], A[:, 1]), ("CT1", A[:, 2], A[:, 3])):
    print("%-6s %10.3f %10.3f %+10.3f %18.0f%% of slices"
          % (name, np.median(a), np.median(b), np.median(b - a), 100 * (b > a).mean()))

fig, ax = plt.subplots(1, 2, figsize=(9, 4))
for k, (name, a, b) in enumerate((("T1", A[:, 0], A[:, 1]), ("CT1", A[:, 2], A[:, 3]))):
    ax[k].scatter(a, b, s=8, alpha=0.6)
    ax[k].plot([-0.2, 1], [-0.2, 1], "k--", lw=0.8)
    ax[k].set_xlabel("correlation with reference, " + OLD_NAME)
    ax[k].set_ylabel("correlation with reference, " + NEW_NAME)
    ax[k].set_title(name + ": points above the line improved")
    ax[k].set_xlim(-0.2, 1); ax[k].set_ylim(-0.2, 1); ax[k].grid(alpha=0.3)
plt.tight_layout(); plt.show()